Design an L-shaped house with as much square footage (floor area) as possible, within the limitations imposed by city codes and certain aesthetic considerations. A rectangular second story will be built above the largest part of the first floor, extending from the back wall but only overlapping half of the front wing. The dimensions of the front wing are x and y, where x is the width. The dimensions of the back wing are a and b, where b is the width. The total width of the house is b.
Constraints
1. The foundation must encompass no more than 3000 square feet.
2. The width of the front wing must be within one-third to one-half of the total house width.
3. Leave at least 1,500 square feet in the inside corner of the L for a pool and patio.
4. The house will sit on a 90 x 150-foot lot with 10 foot minimum setbacks on either side and 25 foot minimum setbacks front and back.
5. The front and back wings must not be disproportionately sized; that is, the length of the back wing should be greater than half of the length of the front wing.
Develop the mathematical optimization model for this problem and Solve using Pyomo.

First, let's define the variables and constraints for our model:

Variables:
x: Width of the front wing

y: Length of the front wing

a: Length of the back wing

b: Width of the back wing

Constraints:
Foundation Area Constraint: xy+ab≤3000

Width Proportion Constraint: 1/3𝑏 ≤ 𝑥 ≤ 1/2𝑏

Pool and Patio Constraint: The area in the inside corner of the L:
(y−b)*a≥1500

Lot Size and Setbacks: The house must fit within the lot after accounting for setbacks: b≤90−2×10=70
y+a≤150−2×25=100

Proportional Sizing:
a≥ 1/2 y

Objective:
Maximize the total floor area, which includes both the first and second floors. The second floor overlaps half of the front wing:

Total Floor Area=xy+ab+1/2xy

Now, we will set up and solve this optimization problem using Pyomo.

In [10]:
!apt-get install -y -qq coinor-libipopt1 coinor-libipopt-dev
!pip install pyomo
import pyomo.environ as pyomo

!wget -N -q "https://ampl.com/dl/open/ipopt/ipopt-linux64.zip"
!unzip -o -q ipopt-linux64

!apt-get install -y -qq coinor-cbc

E: Package 'coinor-libipopt1' has no installation candidate
[ipopt-linux64.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of ipopt-linux64 or
        ipopt-linux64.zip, and cannot find ipopt-linux64.ZIP, period.


In [9]:
# Model definition
model = pyomo.ConcreteModel();

# Variable definition
model.x = pyomo.Var(domain=pyomo.PositiveReals,doc = 'Width of the front wing');
model.y = pyomo.Var(domain=pyomo.PositiveReals,doc = 'Length of the front wing');
model.b = pyomo.Var(domain=pyomo.PositiveReals,doc = 'Width of the back wing');
model.a = pyomo.Var(domain=pyomo.PositiveReals,doc = 'Length of the back wing');
model.z = pyomo.Var(domain=pyomo.PositiveReals,doc = 'Length of the second floor');
model.A1 = pyomo.Var(domain=pyomo.PositiveReals,doc = 'Floor space of the first floor');
model.A2 = pyomo.Var(domain=pyomo.PositiveReals,doc = 'Floor space of the second floor');

# Objective functions
model.obj = pyomo.Objective(expr = model.A1 + model.A2,sense = pyomo.maximize);

# Constraint definition
def rule1a(model):
  return model.A1 == model.x*model.y + model.b*model.a
model.eq1a = pyomo.Constraint(rule = rule1a,doc = 'Floor space of the first floor');
def rule1b(model):
  return model.A2 == model.x*model.z
model.eq1b = pyomo.Constraint(rule = rule1b,doc = 'Floor space of the second floor');

def rule2(model):
  return model.A1 <= 3000
model.eq2 = pyomo.Constraint(rule = rule2,doc = 'Upper limit on floor space of the first floor');

def rule3a(model):
  return model.x >= model.b/3
model.eq3a = pyomo.Constraint(rule = rule3a,doc = 'Lower limit on width of the first floor');
def rule3b(model):
  return model.x <= model.b/2
model.eq3b = pyomo.Constraint(rule = rule3b,doc = 'Upper limit on width of the second floor');

def rule4(model):
  return model.y*(model.b - model.x) >= 1500
model.eq4 = pyomo.Constraint(rule = rule4,doc = 'Lower limit on free space for pool and patio');

def rule5a(model):
  return model.b <= 90 - 2*10
model.eq5a = pyomo.Constraint(rule = rule5a,doc = 'Lower limit on width of the house, accounting for setbacks');
def rule5b(model):
  return model.y + model.a <= 150 - 2*25
model.eq5b = pyomo.Constraint(rule = rule5b,doc = 'Upper limit on width of the house, accounting for setbacks');

def rule6(model):
  return model.a >= model.y/2
model.eq6 = pyomo.Constraint(rule = rule6,doc = 'Upper limit on length of back wing, for aesthetic appeal');

def rule7(model):
  return model.z == model.a + (model.y/2)
model.eq7 = pyomo.Constraint(rule = rule7,doc = 'Upper limit on length of back wing, for aesthetic appeal');

# Solve statement
results = pyomo.SolverFactory('ipopt',executable = '/content/ipopt').solve(model);
#results = pyomo.SolverFactory('cbc').solve(model)

# Printing results
results.write()

print("\n RESULTS \n");
print("Width of the front wing (x) = ",model.x()," ft \n");
print("Length of the front wing (y) = ",model.y()," ft \n");
print("Width of the back wing (b) = ",model.b()," ft \n");
print("Length of the back wing (a) = ",model.a()," ft \n");

print("Length of the second floor (z) = ",model.z()," ft \n");

print("Area of the first floor (A1) = ",model.A1()," ft \n");
print("Area of the second floor (A2) = ",model.A2()," ft \n");
print("Total floor space (A1 + A2) = ",model.obj()," sq. ft \n");

Failed to set executable for solver ipopt. File with name=/content/ipopt either does not exist or it is not executable. To skip this validation, call set_executable with validate=False.
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/pyomo/opt/base/solvers.py", line 148, in __call__
    opt = self._cls[_name](**kwds)
  File "/usr/local/lib/python3.10/dist-packages/pyomo/solvers/plugins/solvers/IPOPT.py", line 42, in __init__
    super(IPOPT, self).__init__(**kwds)
  File "/usr/local/lib/python3.10/dist-packages/pyomo/opt/solver/shellcmd.py", line 66, in __init__
    self.set_executable(name=executable, validate=validate)
  File "/usr/local/lib/python3.10/dist-packages/pyomo/opt/solver/shellcmd.py", line 115, in set_executable
    raise ValueError(
ValueError: Failed to set executable for solver ipopt. File with name=/content/ipopt either does not exist or it is not executable. To skip this validation, call set_executable with validate=False.


RuntimeError: Attempting to use an unavailable solver.

The SolverFactory was unable to create the solver "ipopt"
and returned an UnknownSolver object.  This error is raised at the point
where the UnknownSolver object was used as if it were valid (by calling
method "solve").

The original solver was created with the following parameters:
	executable: /content/ipopt
	type: ipopt
	_args: ()
	options: {}